# Kaggle Playground Competition Season 6 - Episode 8: Predicting Smartphone Addiction
This notebook covers an end-to-end approach for [Kaggle's August 2026 playground competition](https://www.kaggle.com/competitions/playground-series-s6e8/), **Predicting Smartphone Addiction**. My personal goal with this competition, as per usual, is to keep my own data science skills sharp and perhaps some new things along the way. In full transparency, I am using ChatGPT to help me as a **thought partner**, NOT as an "end-to-end solver". (No Codex / Claude Code being used here!) Even in this case, I have zero expectation of winning this competition, and I'm completely fine with that!

This being a Kaggle playground competition, the dataset supporting this competition is one synthesized based on real dataset using the same raw data features. As the competition name implies, we are attempting to predict smartphone addiction as a **binary classification problem**: either the person is addicted or is not addicted. We will do a deeper dive analysis of the supporting data in the Exploratory Data Analysis section.

## 0: Notebook Setup
Let's set ourselves up for success by doing all the necessary "administrative overhead" for our work!

In [ ]:
# Importing the necessary Python libraries
import polars as pl

In [ ]:
# Loading in the raw data
df_train_raw = pl.read_csv('data/raw/train.csv')
df_test_raw = pl.read_csv('data/raw/test.csv')

## 1: Exploratory Data Analysis
Let's jump into our notebook by performing an **exploratory data analysis**. We are going to approach this from two perspectives. First, we're going to get a semantic understanding. As you'll see looking at the dataset, it's hard to get an understanding of what we're working with solely based on the raw feature (column) names. We'll use Kaggle's platform to help us get a better understanding of the data at a semantic level. Second, we're going to do a deep dive data analysis, which also includes some statistical analyses. By doing this analysis from both of these perspectives, we will gain a robust understanding of the data we are working with and start to strategize on how feature engineering might start to take form.

### 1.1: Understanding the data at a semantic level
As we noted at the top of the notebook, Kaggle's playground competitions are comprised of datasets that are synthesized based on other datasets, so generally speaking, Kaggle doesn't restate what the data is and instead points to the data it was originally sourced from. As of August 1, 2026, it looks like Kaggle mistakenly shared a bad link, meaning that we don't actually have a way to truly determine which dataset this playground competition dataset is sourced from. In combing the discussion board, it appears that this playground dataset is originally sourced from [this dataset](https://www.kaggle.com/datasets/jayjoshi37/smartphone-usage-and-addiction-prediction/data). Unfortunately, not even this dataset covers everything. Specifically, it does not cover `stress_level` nor `academic_work_impact`, so we're going to have to make an informative guess as to what those mean.

Let's get the semantic understanding for each of these respective data features.

- `id`: The unique identifier for each record assigned by Kaggle. (It is not important to the semantic understanding of the overall dataset, and we will not use it when constructing the final model.)
- `age`: The age of the user. Range is 18-35 years old.
- `daily_screen_time_hours`: The average number of hours a user spends on their smartphone each day.
- `social_media_hours`: The average number of hours a user spends on social media each day.
- `gaming_hours`: The average number of hours a user plays mobile games on their smartphone each day.
- `work_study_hours`: The average number of hours a user is productive throughout the day, whether that be actively working or studying. 
- `sleep_hours`: The average number of hours a user sleeps each day.
- `notifications_per_day`: The number of notifications received daily. 
- `app_opens_per_day`: The number apps open daily.
- `weekend_screen_time`: The average amount of hours a user is on their smartphone on the weekend.
- `gender`: The gender the user idenitifies as. There are three categories present in the dataset: "Male", "Female", and "Other".
- `stress_level`: The stress level the user identifies with. There are three categories present in the dataset: "Low", "Medium", and "High".
- `academic_work_impact`: The impact a user experiences based on academic work. This is a boolean "Yes" or "No" field.
- `addicted_label`: The target variable represented by a binary 1 or 0. 1 = yes, the user is addicted; 0 = no, the user is not addicted.

A few general notes about these data elements:
- There are a number of times where the word "average" is used. It is unclear what they mean by that, so I'm going to assume it's the mean average.
- There are a couple features like `notifications_per_day` where all the data records are represented by integer values. The reason I find this curious is because this almost certainly can't represent a mean average values since means are rarely integers, especially when all the records are integers. I'm not sure what to think about this, so we're just going to assume it's something close to a mean average.
- As noted above, those last two features I could not find present in other potential datasets this playground dataset what synthesized on. `stress_level` seemed pretty self explanatory, but I am really not sure what `academic_work_impact` means. Not sure if this means that the user is focused on academic work in a positive or negative way. By positive, I mean that the user is positively using their smartphone for research purposes. By negative, I mean that the user may be using their smartphone as an "escape" to relax or procrastinate from their academic work. I don't know! We're just going to have to roll with what it is.

In [5]:
df_train_raw.head()

id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact,addicted_label
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,i64
0,24.0,null,1.83,1.59,2.11,7.46,122.0,38.0,8.63,"""Male""","""Medium""","""No""",1
1,19.0,5.97,1.08,null,3.03,8.22,76.0,19.0,null,"""Female""","""Medium""","""No""",0
2,18.0,5.09,null,null,null,6.25,134.0,60.0,7.47,"""Female""","""Low""","""Yes""",0
3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,"""Other""","""Low""",null,1
4,26.0,11.2,1.87,2.81,1.95,5.25,null,null,13.39,"""Female""","""Medium""","""No""",1
